# Model benchmark (A0 only): which model do we freeze for the project? (BigCodeBench-Hard + HumanEval+)

**Question.** On BigCodeBench-Hard (148 tasks, hard) and HumanEval+ (164 tasks, easy), how does each candidate model do **alone** (one call, no builder, no planner, no tests, no repair)? The answer decides which model the whole team freezes for the pipeline experiments.

**What is compared:** only the model. Everything else is frozen and identical for every model (see cell 2): same tasks, same order, same prompt, greedy decoding, same token limit, same scoring (official hidden tests).

**Outputs** (in `/kaggle/working/results`):
- `A0__<model>__<version>.jsonl` (BigCodeBench) and `A0__<model>__humaneval_plus__<version>.jsonl` (HumanEval+): one line per task (code, raw output, tokens, latency, finish reason)
- `<model>__<benchmark>__hidden.json`: pass/fail on the hidden tests
- `frozen_config.json`: all frozen settings, library versions, model revisions
- `metrics_input_bigcodebench_hard.jsonl` and `metrics_input_humaneval_plus.jsonl`: same results in the format of `metrics.py` (system = model name). Run `metrics.py` once per file.

**Note (v2):** some models (DeepSeek) answer like a completion (indented block or only the function body). `normalize_code` repairs this for ALL models equally, and the `reformatted_%` column shows how often it was needed. The raw model answer is always kept in `raw_output`.

**Models too big for the GPU** (for example the 32B) run through a hosted API (OpenRouter) with the same frozen settings. Their latency includes the network and the provider may quantize the model, so say it in the report.

**Each model is loaded once and runs both benchmarks** (set `BENCHMARKS` in cell 2 to run only one). The prompt is frozen *per benchmark*: HumanEval+ uses the team's short prompt, BigCodeBench-Hard the longer docstring-oriented one.

**How to use:** Accelerator = GPU T4 x2, Internet ON. Run cell 1, then *Run > Restart session*, then cells 2 to 8 in order. Cell 6 is resumable: if the session stops, rerun it and finished tasks are skipped. Download `/kaggle/working/results` before the session ends.

In [1]:
# 1. INSTALL (light). Do NOT pip install the full bigcodebench: it pulls vllm.
!pip install -q evalplus
!pip install -q --no-deps bigcodebench
!pip install -q appdirs datasets fire gradio_client multipledispatch pqdm rich termcolor tree-sitter tree-sitter-python wget tempdir e2b httpx accelerate bitsandbytes
print("Installed. Now: Run > Restart session, then continue from cell 2.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-tools 1.85.1 requires protobuf<8.0.0,>=7.35.1, but you have protobuf 5.29.6 which is incompatible.
kaggle-environments 1.33.3 requires protobuf>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
onnx 1.23.2 requires protobuf>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-texttospeech 2.38.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible

In [2]:
# 2. MODELS + FROZEN SETTINGS  (the only cell you normally edit)
import os, json, re, ast, sys, time, random, math, gc, shutil, collections
from dataclasses import dataclass, asdict

# ---- candidate models. quant: "fp16" or "4bit" (NF4). T4 x2 = 2 x 15 GB, so >= 14B must be 4-bit.
MODELS = [
    {"name": "qwen2.5-coder-1.5b",  "hf": "Qwen/Qwen2.5-Coder-1.5B-Instruct",     "quant": "fp16"},
    {"name": "qwen2.5-coder-3b",    "hf": "Qwen/Qwen2.5-Coder-3B-Instruct",       "quant": "fp16"},
    {"name": "qwen2.5-coder-7b",    "hf": "Qwen/Qwen2.5-Coder-7B-Instruct",       "quant": "fp16"},
    {"name": "qwen2.5-coder-14b",   "hf": "Qwen/Qwen2.5-Coder-14B-Instruct",      "quant": "4bit"},
    {"name": "deepseek-coder-6.7b", "hf": "deepseek-ai/deepseek-coder-6.7b-instruct", "quant": "fp16"},
    # Too big for 2 x T4 (the 32B weights are ~65 GB to download): use a hosted API instead.
    # Needs a Kaggle Secret named OPENROUTER_KEY (Add-ons > Secrets). The key is NEVER written in the notebook.
    {"name": "qwen2.5-coder-32b-api", "hf": "qwen/qwen-2.5-coder-32b-instruct", "quant": "api", "backend": "openrouter"},
]
# By default only the LOCAL models run. Add "qwen2.5-coder-32b-api" to run the hosted one.
MODELS_TO_RUN   = [m["name"] for m in MODELS if not m.get("backend")]   # e.g. ["qwen2.5-coder-7b"] for a quick trial
REFERENCE_MODEL = "qwen2.5-coder-7b"             # model used as the reference in paired comparisons
BENCHMARKS      = ["bigcodebench-hard", "humaneval+"]   # which benchmarks to run (one or both)
N_TASKS         = None                           # None = all tasks of each benchmark. Use 5 for a quick trial run
DELETE_CACHE_AFTER = True                        # delete each model's weights after its run (Kaggle disk is limited)

# ---- FROZEN: identical for every model. Changing any of this = a new prompt_version = new result files.
SUBSET, PROMPT_SPLIT, EXECUTION = "hard", "complete", "gradio"   # BigCodeBench-Hard, docstring prompt, hosted scorer
SPLIT, DEV_SIZE, SPLIT_SEED = "all", 20, 1234

@dataclass(frozen=True)
class Frozen:
    system: str = "A0"                 # one call, no builder / planner / tests / repair
    do_sample: bool = False            # greedy decoding
    repetition_penalty: float = 1.0    # set explicitly: model generation_config files may default to another value
    max_new_tokens: int = 768          # same as the team notebook; truncation rate is reported per model
    prompt_version: str = "a0_models_v2"   # v2 = normalize_code (dedent + body wrapping) added
    seed: int = 1234
FROZEN = Frozen()

SYSTEM_BCB = """You are an expert Python programmer. Reply with exactly ONE ```python code block containing all imports, any helper functions from the task, and the complete function with the exact name and signature given in the task. Do NOT write docstrings, comments, example usage or test code. No explanation outside the code block.

FOLLOW THE DOCSTRING LITERALLY:
- Return exactly the type and structure the docstring documents (a tuple is not an array; plain strings are not objects like IPv4Address; use the stated DataFrame columns, order and dtypes).
- Raise exactly the exceptions listed under 'Raises', under exactly the stated conditions, and validate inputs BEFORE using them (check first, then look up).
- Respect boundary conventions: inclusive vs exclusive ends, exact number of items, and never silently skip or drop elements the task asks you to process.
- Use the libraries and functions the task names; do not substitute others.
- Do not add behaviour the docstring does not ask for."""

# HumanEval+ prompt = the team's A0 prompt (same as in the team's HumanEval+ notebook)
SYSTEM_HE = ("You are an expert Python programmer. Reply with exactly ONE ```python code block containing "
             "all imports, any helper functions from the task, and the complete function with the exact name "
             "and signature given in the task. Do NOT write docstrings, comments, example usage or test code. "
             "No explanation outside the code block.")

OUT_DIR = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(FROZEN.seed)
print("Frozen settings:", asdict(FROZEN))

Frozen settings: {'system': 'A0', 'do_sample': False, 'repetition_penalty': 1.0, 'max_new_tokens': 768, 'prompt_version': 'a0_models_v2', 'seed': 1234}


In [3]:
# 3. HELPERS: tasks, code extraction, statistics, hidden scoring (same logic as the team's BigCodeBench and HumanEval+ notebooks)
@dataclass
class Task:
    task_id: str
    prompt: str                      # what the model sees (never the hidden tests)
    entry_point: str = "task_func"

def _trim(block: str) -> str:
    """Remove blank lines at the start and the end, but KEEP the indentation of the first code line.
    (A plain .strip() would remove the indent of line 1 only, which creates 'unexpected indent' errors.)"""
    lines = block.split("\n")
    while lines and not lines[0].strip():
        lines.pop(0)
    while lines and not lines[-1].strip():
        lines.pop()
    return "\n".join(l.rstrip() for l in lines)

def extract_code(text: str) -> str:
    """Take the code from a markdown block, even when the closing fence is missing."""
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return _trim(max(blocks, key=len))
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return _trim(m.group(1))
    return _trim(text)

def syntax_error(code: str):
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"
    except Exception as e:
        return f"{type(e).__name__}: {e}"

import textwrap

def normalize_code(code: str, task):
    """Some models answer like a completion: the whole block is indented, or only the function body is given.
    1) remove the common indentation, 2) if the function is missing, rebuild it from the task prompt + the body.
    Applied to ALL models in the same way (a no-op when the code is already a normal function).
    Returns (code, how) where how is None, "dedent", "body_wrapped" or "empty"."""
    how = None
    if not code.strip():
        return code, "empty"
    d = textwrap.dedent(code)
    if d != code:
        code, how = d, "dedent"
    if not re.search(rf"^\s*def\s+{re.escape(task.entry_point)}\s*\(", code, re.M):
        body = "\n".join(("    " + l if l.strip() else l) for l in code.splitlines())
        code, how = task.prompt.rstrip() + "\n" + body, "body_wrapped"
    return code, how

def wilson(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a) / d, (c + a) / d)

def sign_test(b: int, c: int) -> float:
    """Exact two-sided sign test (= exact McNemar) on discordant pairs."""
    m = b + c
    if m == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(0, min(b, c) + 1)) / 2 ** m)

def read_jsonl(path):
    return [json.loads(l) for l in open(path) if l.strip()] if os.path.isfile(path) else []

def slug(bench):
    return bench.replace("+", "_plus").replace("-", "_")

def load_bcb_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, prompt=p[key], entry_point=p.get("entry_point", "task_func"))
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

def score_bcb(solutions: dict, work_dir: str, calibrated=True) -> dict:
    """{task_id: bool} from the HIDDEN BigCodeBench tests (official evaluator)."""
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples = os.path.join(work_dir, "samples.jsonl")
    path = samples.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples, "w") as f:
            for tid, c in items.items():
                f.write(json.dumps({"task_id": tid, "solution": c or "pass"}) + "\n")
        if os.path.isfile(path):
            os.remove(path)          # evaluate() reuses an old results file if it finds one

    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples, execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        print("selective_evaluate failed (" + str(e)[:200] + ") -> retrying with all tasks padded")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid) for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples, execution=EXECUTION, calibrated=calibrated)
    if not os.path.isfile(path):
        raise RuntimeError("BigCodeBench produced no results file at " + path)
    data = json.load(open(path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

def load_he_tasks():
    from evalplus.data import get_human_eval_plus
    tasks = [Task(task_id=tid, prompt=p["prompt"], entry_point=p["entry_point"])
             for tid, p in get_human_eval_plus().items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

def score_he(solutions: dict, work_dir: str) -> dict:
    """{task_id: bool} from the HIDDEN HumanEval+ tests (EvalPlus: original tests AND the extra "plus" tests).
    EvalPlus needs all 164 problems in the samples file, so tasks we did not run are padded with "pass" and ignored."""
    import glob, hashlib, subprocess
    from evalplus.data import get_human_eval_plus
    os.makedirs(work_dir, exist_ok=True)
    full = {tid: "pass" for tid in get_human_eval_plus()}
    full.update({tid: (c or "pass") for tid, c in solutions.items()})
    key = hashlib.md5(json.dumps(full, sort_keys=True).encode()).hexdigest()[:10]
    path = os.path.join(work_dir, f"samples_{key}.jsonl")
    found = glob.glob(os.path.join(work_dir, f"samples_{key}*result*.json"))
    if not found:                                   # reuse a previous evaluation of the exact same solutions
        with open(path, "w") as f:
            for tid, c in full.items():
                f.write(json.dumps({"task_id": tid, "solution": c}) + "\n")
        p = subprocess.run([sys.executable, "-m", "evalplus.evaluate", "--dataset", "humaneval",
                            "--samples", path, "--i-just-wanna-run"], capture_output=True, text=True)
        found = glob.glob(os.path.join(work_dir, f"samples_{key}*result*.json"))
        if not found:
            print("EVALPLUS FAILED\nSTDOUT:", p.stdout[-2000:], "\nSTDERR:", p.stderr[-2000:])
            raise RuntimeError("evalplus produced no results file")
    data = json.load(open(found[0]))["eval"]
    out = {}
    for tid in solutions:
        r = data.get(tid, [{}])[0]
        out[tid] = (r.get("base_status") == "pass") and (r.get("plus_status") == "pass")
    return out

def select_tasks(tasks, limit=None):
    return tasks[:limit] if limit else tasks

LOADERS = {"bigcodebench-hard": load_bcb_tasks, "humaneval+": load_he_tasks}
SCORERS = {"bigcodebench-hard": score_bcb, "humaneval+": score_he}
SYSTEMS = {"bigcodebench-hard": SYSTEM_BCB, "humaneval+": SYSTEM_HE}

TASKS_BY_BENCH = {}
for b in BENCHMARKS:
    _all = LOADERS[b]()
    TASKS_BY_BENCH[b] = select_tasks(_all, N_TASKS)
    print(f"{b}: {len(_all)} tasks, {len(TASKS_BY_BENCH[b])} selected")

README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/v0.1.0_hf-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  551kB            

data/v0.1.0_hf-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/v0.1.1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  551kB            

data/v0.1.1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/v0.1.2-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  552kB            

... (6 outputs left out)

Generating v0.1.3 split:   0%|          | 0/148 [00:00<?, ? examples/s]

Generating v0.1.4 split:   0%|          | 0/148 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

bigcodebench-hard: 148 tasks, 148 selected
humaneval+: 164 tasks, 164 selected


In [4]:
# 4. CHECK THE SCORING FIRST (reference solutions must pass ~100%). If not, the setup is wrong, not the models.
def reference_solutions(bench, n=5):
    if bench == "bigcodebench-hard":
        from bigcodebench.data import get_bigcodebench
        probs = get_bigcodebench(subset=SUBSET)
        ids = list(probs)[:n]
        return {i: probs[i]["complete_prompt"] + "\n" + probs[i]["canonical_solution"] for i in ids}
    from evalplus.data import get_human_eval_plus
    probs = get_human_eval_plus()
    ids = list(probs)[:n]
    return {i: probs[i]["prompt"] + probs[i]["canonical_solution"] for i in ids}

for b in BENCHMARKS:
    _ref = reference_solutions(b)
    _res = SCORERS[b](_ref, f"{OUT_DIR}/_check_{slug(b)}")
    print(b, ":", sum(_res.values()), "/", len(_ref), "reference solutions pass", _res)

Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 1.000
pass@1:	1.000
bigcodebench-hard : 5 / 5 reference solutions pass {'BigCodeBench/13': True, 'BigCodeBench/15': True, 'BigCodeBench/17': True, 'BigCodeBench/19': True, 'BigCodeBench/34': True}
humaneval+ : 5 / 5 reference solutions pass {'HumanEval/0': True, 'HumanEval/1': True, 'HumanEval/2': True, 'HumanEval/3': True, 'HumanEval/4': True}


In [5]:
# 5. MODEL RUNNER: one model at a time, A0 only, frozen settings
def build_text(tok, system, user):
    """Use the model's own chat template. If it rejects a system role, merge it into the user message."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        msgs = [{"role": "user", "content": system + "\n\n" + user}]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def make_hf_generate(spec):
    """Load the model once. Returns (generate_fn, cleanup_fn, info)."""
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(spec["hf"])
    kw = dict(device_map="auto")
    if spec["quant"] == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16          # T4 does not support bfloat16
    model = AutoModelForCausalLM.from_pretrained(spec["hf"], **kw).eval()
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    info = {"hf": spec["hf"], "quant": spec["quant"], "revision": getattr(model.config, "_commit_hash", None),
            "generation_config_defaults": model.generation_config.to_dict()}
    print("Loaded", spec["name"], "| revision", info["revision"], "| device", model.device)

    def generate(system, user):
        text = build_text(tok, system, user)
        inputs = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)   # the template already adds BOS if needed
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=FROZEN.max_new_tokens, do_sample=FROZEN.do_sample,
                                 repetition_penalty=FROZEN.repetition_penalty,
                                 temperature=None, top_p=None, top_k=None, pad_token_id=pad_id)
        gen = out[0][inputs.input_ids.shape[1]:]
        return {"reply": tok.decode(gen, skip_special_tokens=True),
                "prompt_tokens": int(inputs.input_ids.shape[1]), "gen_tokens": int(len(gen)),
                "latency": round(time.time() - t0, 2),
                "finish_reason": "length" if len(gen) >= FROZEN.max_new_tokens else "stop"}

    def cleanup():
        nonlocal model
        del model
        gc.collect()
        torch.cuda.empty_cache()

    return generate, cleanup, info

def make_api_generate(spec):
    """Hosted model through OpenRouter (OpenAI-compatible API). The key comes from Kaggle Secrets or an env variable."""
    import requests
    key = os.environ.get("OPENROUTER_KEY")
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("OPENROUTER_KEY")
        except Exception:
            key = None
    if not key:
        raise RuntimeError("Missing OPENROUTER_KEY: add it in Kaggle (Add-ons > Secrets) and attach it to this notebook")
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": "Bearer " + key}

    def generate(system, user):
        payload = {"model": spec["hf"], "temperature": 0, "max_tokens": FROZEN.max_new_tokens,
                   "repetition_penalty": FROZEN.repetition_penalty,
                   "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
        t0 = time.time()
        for attempt in range(5):                      # retry on rate limit / server errors
            resp = requests.post(url, headers=headers, json=payload, timeout=300)
            if resp.status_code in (429, 500, 502, 503, 504):
                time.sleep(3 * 2 ** attempt)
                continue
            break
        resp.raise_for_status()
        data = resp.json()
        choice, usage = data["choices"][0], data.get("usage") or {}
        return {"reply": choice["message"].get("content") or "",
                "prompt_tokens": int(usage.get("prompt_tokens", 0)), "gen_tokens": int(usage.get("completion_tokens", 0)),
                "latency": round(time.time() - t0, 2),
                "finish_reason": "length" if choice.get("finish_reason") == "length" else "stop",
                "provider": data.get("provider")}

    info = {"hf": spec["hf"], "quant": "api", "revision": None, "generation_config_defaults": None,
            "note": "hosted model: the provider may quantize it and sets its own defaults; latency includes the network"}
    print("API model ready:", spec["name"])
    return generate, (lambda: None), info

def make_generate(spec):
    return make_api_generate(spec) if spec.get("backend") == "openrouter" else make_hf_generate(spec)

def run_model(spec, tasks, generate, bench, out_dir=OUT_DIR):
    """Resumable: finished tasks are read back from the jsonl file and skipped."""
    suffix = "" if bench == "bigcodebench-hard" else f"__{slug(bench)}"      # BigCodeBench keeps the v2 file name
    path = f"{out_dir}/A0__{spec['name']}{suffix}__{FROZEN.prompt_version}.jsonl"
    done = {r["task_id"] for r in read_jsonl(path)}
    todo = [t for t in tasks if t.task_id not in done]
    print(f"{spec['name']} on {bench}: {len(done)} done, {len(todo)} to run")
    t_start = time.time()
    for k, t in enumerate(todo, 1):
        base = {"task_id": t.task_id, "system": spec["name"], "model": spec["hf"], "quant": spec["quant"],
                "prompt_version": FROZEN.prompt_version, "benchmark": bench}
        try:
            g = generate(SYSTEMS[bench], t.prompt)
            code_, how = normalize_code(extract_code(g["reply"]), t)
            rec = {**base, "final_code": code_, "reformatted": how, "provider": g.get("provider"), "raw_output": g["reply"], "tokens": g["prompt_tokens"] + g["gen_tokens"],
                   "gen_tokens": g["gen_tokens"], "latency": g["latency"], "finish_reason": g["finish_reason"],
                   "syntax_error": syntax_error(code_), "error": None}
        except Exception as e:          # one bad task (e.g. out of memory) must not kill the run
            try:
                import torch; torch.cuda.empty_cache()
            except Exception:
                pass
            rec = {**base, "final_code": "", "reformatted": None, "provider": None, "raw_output": "", "tokens": 0, "gen_tokens": 0, "latency": 0.0,
                   "finish_reason": "error", "syntax_error": None, "error": f"{type(e).__name__}: {str(e)[:200]}"}
        with open(path, "a") as f:
            f.write(json.dumps(rec) + "\n")
        el = time.time() - t_start
        print(f"  {spec['name']} [{k}/{len(todo)}] {t.task_id} | {rec['gen_tokens']} tok | {rec['finish_reason']} "
              f"| elapsed {el/60:.0f} min | ~{el/k*(len(todo)-k)/60:.0f} min left", flush=True)
    keep = {t.task_id for t in tasks}
    return [r for r in read_jsonl(path) if r["task_id"] in keep]

def delete_hf_cache(hf_name):
    from huggingface_hub.constants import HF_HUB_CACHE
    d = os.path.join(HF_HUB_CACHE, "models--" + hf_name.replace("/", "--"))
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True)
        print("Deleted cache:", d)

def library_versions():
    from importlib.metadata import version, PackageNotFoundError
    out = {}
    for p in ["torch", "transformers", "bitsandbytes", "accelerate", "bigcodebench", "evalplus"]:
        try:
            out[p] = version(p)
        except PackageNotFoundError:
            out[p] = None
    return out

In [6]:
# 6. RUN ALL MODELS (resumable: rerun this cell if it stops, finished tasks are kept)
# Each model is loaded ONCE, then runs every benchmark in BENCHMARKS (saves download and loading time).
RECORDS = {b: {} for b in BENCHMARKS}
RUN_INFO = {}
for spec in MODELS:
    if spec["name"] not in MODELS_TO_RUN:
        continue
    generate, cleanup, info = make_generate(spec)
    RUN_INFO[spec["name"]] = info
    try:
        for b in BENCHMARKS:
            RECORDS[b][spec["name"]] = run_model(spec, TASKS_BY_BENCH[b], generate, b)
    finally:
        cleanup()
        if DELETE_CACHE_AFTER and not spec.get("backend"):
            delete_hf_cache(spec["hf"])
    json.dump({"frozen": asdict(FROZEN), "bcb_subset": SUBSET, "bcb_prompt_split": PROMPT_SPLIT,
               "benchmarks": {b: {"n_tasks": len(TASKS_BY_BENCH[b]), "system_prompt": SYSTEMS[b]} for b in BENCHMARKS},
               "libraries": library_versions(), "models": RUN_INFO},
              open(f"{OUT_DIR}/frozen_config.json", "w"), indent=1)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (63 outputs left out)

<unknown>:6: SyntaxWarning: invalid escape sequence '\W'


  deepseek-coder-6.7b [91/148] BigCodeBench/678 | 401 tok | stop | elapsed 30 min | ~19 min left
  deepseek-coder-6.7b [92/148] BigCodeBench/720 | 273 tok | stop | elapsed 30 min | ~19 min left
  deepseek-coder-6.7b [93/148] BigCodeBench/723 | 310 tok | stop | elapsed 31 min | ~18 min left
  deepseek-coder-6.7b [94/148] BigCodeBench/752 | 284 tok | stop | elapsed 31 min | ~18 min left
  deepseek-coder-6.7b [95/148] BigCodeBench/760 | 325 tok | stop | elapsed 32 min | ~18 min left
  deepseek-coder-6.7b [96/148] BigCodeBench/763 | 279 tok | stop | elapsed 32 min | ~17 min left
  deepseek-coder-6.7b [97/148] BigCodeBench/765 | 149 tok | stop | elapsed 32 min | ~17 min left
  deepseek-coder-6.7b [98/148] BigCodeBench/771 | 359 tok | stop | elapsed 32 min | ~16 min left
  deepseek-coder-6.7b [99/148] BigCodeBench/777 | 312 tok | stop | elapsed 33 min | ~16 min left
  deepseek-coder-6.7b [100/148] BigCodeBench/785 | 204 tok | stop | elapsed 33 min | ~16 min left
  deepseek-coder-6.7b [101/14

<unknown>:7: SyntaxWarning: invalid escape sequence '\W'


  deepseek-coder-6.7b [104/148] BigCodeBench/854 | 179 tok | stop | elapsed 34 min | ~14 min left
  deepseek-coder-6.7b [105/148] BigCodeBench/857 | 296 tok | stop | elapsed 34 min | ~14 min left
  deepseek-coder-6.7b [106/148] BigCodeBench/865 | 136 tok | stop | elapsed 34 min | ~14 min left
  deepseek-coder-6.7b [107/148] BigCodeBench/870 | 327 tok | stop | elapsed 35 min | ~13 min left
  deepseek-coder-6.7b [108/148] BigCodeBench/879 | 278 tok | stop | elapsed 35 min | ~13 min left
  deepseek-coder-6.7b [109/148] BigCodeBench/897 | 239 tok | stop | elapsed 35 min | ~13 min left
  deepseek-coder-6.7b [110/148] BigCodeBench/906 | 287 tok | stop | elapsed 36 min | ~12 min left
  deepseek-coder-6.7b [111/148] BigCodeBench/914 | 590 tok | stop | elapsed 36 min | ~12 min left
  deepseek-coder-6.7b [112/148] BigCodeBench/915 | 233 tok | stop | elapsed 37 min | ~12 min left
  deepseek-coder-6.7b [113/148] BigCodeBench/916 | 261 tok | stop | elapsed 37 min | ~11 min left
  deepseek-coder-6.7

In [7]:
# 7. SCORE WITH THE HIDDEN TESTS, BUILD THE TABLE, EXPORT FOR metrics.py
import pandas as pd

def summarize_model(recs, hidden):
    n = len(recs)
    passed = sum(1 for r in recs if hidden.get(r["task_id"]))
    lo, hi = wilson(passed, n)
    gen_tok = sum(r["gen_tokens"] for r in recs)
    lat = sum(r["latency"] for r in recs)
    return {"passed": f"{passed}/{n}", "pass_rate": round(passed / n, 3), "ci95": [round(lo, 3), round(hi, 3)],
            "truncated_%": round(100 * sum(r["finish_reason"] == "length" for r in recs) / n, 1),
            "syntax_err_%": round(100 * sum(bool(r["syntax_error"]) for r in recs) / n, 1),
            "reformatted_%": round(100 * sum(bool(r.get("reformatted")) for r in recs) / n, 1),
            "errors": sum(r["finish_reason"] == "error" for r in recs),
            "avg_gen_tokens": round(gen_tok / n, 1), "avg_latency_s": round(lat / n, 2),
            "tok_per_s": round(gen_tok / lat, 1) if lat else None,
            "tokens_per_solved": round(sum(r["tokens"] for r in recs) / passed, 1) if passed else None}

def export_metrics_input(records, hidden, path):
    with open(path, "w") as f:
        for name, recs in records.items():
            for r in recs:
                f.write(json.dumps({"task_id": r["task_id"], "system": name,
                                    "hidden_pass": bool(hidden[name].get(r["task_id"])), "self_pass": None,
                                    "iters": 0, "tokens": r["tokens"], "latency": r["latency"]}) + "\n")

HIDDEN, SUMMARY, TABLES = {}, {}, {}
for b in BENCHMARKS:
    HIDDEN[b], SUMMARY[b] = {}, {}
    for name, recs in RECORDS[b].items():
        HIDDEN[b][name] = SCORERS[b]({r["task_id"]: r["final_code"] for r in recs}, f"{OUT_DIR}/score_{slug(b)}_{name}")
        json.dump(HIDDEN[b][name], open(f"{OUT_DIR}/{name}__{slug(b)}__hidden.json", "w"), indent=1)
        SUMMARY[b][name] = summarize_model(recs, HIDDEN[b][name])
    TABLES[b] = pd.DataFrame(SUMMARY[b]).T
    print(f"\n=== {b} ===")
    print(TABLES[b].to_string())

    ref = REFERENCE_MODEL if REFERENCE_MODEL in HIDDEN[b] else next(iter(HIDDEN[b]))
    print(f"\nPaired comparison against {ref} on {b} (same tasks):")
    for name in HIDDEN[b]:
        if name == ref:
            continue
        ids = sorted(set(HIDDEN[b][ref]) & set(HIDDEN[b][name]))
        fixed = sum(1 for t in ids if HIDDEN[b][name][t] and not HIDDEN[b][ref][t])
        broken = sum(1 for t in ids if HIDDEN[b][ref][t] and not HIDDEN[b][name][t])
        print(f"  {name}: better on {fixed}, worse on {broken}, sign-test p = {sign_test(fixed, broken):.4f}")

    out_path = f"{OUT_DIR}/metrics_input_{slug(b)}.jsonl"
    export_metrics_input(RECORDS[b], HIDDEN[b], out_path)
    print(f"Exported {out_path}. Then run:  python metrics.py {out_path} --baseline {ref} --out out_{slug(b)}/")

print("\n=== pass rate by model and benchmark ===")
print(pd.DataFrame({b: {n: SUMMARY[b][n]["pass_rate"] for n in SUMMARY[b]} for b in BENCHMARKS}).to_string())

Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.047
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.101
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.189
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.243
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.142

=== bigcodebench-hard ===
                     passed pass_rate            ci95 truncated_% syntax_err_% reformatted_% errors avg_gen_tokens avg

In [8]:
# 8. HOW TO READ THE RESULT (decision checklist for freezing the model)
# - truncated_% above ~3%: the token limit hurts this model; the comparison is unfair. Raise max_new_tokens for ALL models
#   (new prompt_version), do not change it for one model only.
# - syntax_err_%: high values usually mean broken code blocks (format problem), not wrong logic.
# - Look at the CI, not only the rate: with 148 tasks, two models within a few points are statistically tied.
# - Freezing rule: pick the model whose pass rate is NOT saturated (so the pipeline has room to help) and that fits
#   in the GPU and in the time budget for A3 (A3 makes ~5 to 8 calls per task, so it is several times slower than A0).
# - Mixed precision (fp16 vs 4-bit) is a confounder: say it in the report, or also run the 7B in 4-bit to measure the effect.
# - HumanEval+ is easy: a big model may already be near its ceiling there, so a gain from the pipeline is hard to see.
#   Use BigCodeBench-Hard as the main benchmark and HumanEval+ to check that the ranking of the models holds.
for b in BENCHMARKS:
    print("\n" + b)
    print(TABLES[b][["pass_rate", "ci95", "truncated_%", "syntax_err_%", "reformatted_%", "avg_latency_s"]].to_string())


bigcodebench-hard
                    pass_rate            ci95 truncated_% syntax_err_% reformatted_% avg_latency_s
qwen2.5-coder-1.5b      0.047  [0.023, 0.094]         1.4          2.0           0.7         21.02
qwen2.5-coder-3b        0.101   [0.062, 0.16]         6.1          5.4           0.0          28.0
qwen2.5-coder-7b        0.189   [0.134, 0.26]         1.4          2.0           0.0         18.68
qwen2.5-coder-14b       0.243  [0.181, 0.318]         0.0          0.0           0.0          37.4
deepseek-coder-6.7b     0.142  [0.095, 0.207]         0.0          0.0          30.4         19.72

humaneval+
                    pass_rate            ci95 truncated_% syntax_err_% reformatted_% avg_latency_s
qwen2.5-coder-1.5b      0.604  [0.527, 0.675]         0.0          0.6           0.0          3.72
qwen2.5-coder-3b         0.78  [0.711, 0.837]         0.0          0.0           0.0          5.12
qwen2.5-coder-7b        0.811  [0.744, 0.864]         0.0          0.0        